# ESA CCI CMIP7 fix-plan example

This notebook uses a tiny synthetic CMIP7-style ESA CCI water-vapour dataset and a JSON fix plan to show workflow-style reformatting with Woodpecker.

In [ ]:
from pathlib import Path

import numpy as np
import woodpecker_cmip7_plugin  # noqa: F401 - imports plugin fixes for editable installs

import woodpecker
from woodpecker.testing import make_cmip7

Create a deterministic ESA CCI-like input: `prw` should become `tcwv`, `bnds` should become `nv`, selected metadata should be added, and latitude should be increasing.

In [ ]:
source_name = "ESACCI-WATERVAPOUR-L3C-TCWV-meris-005deg-2002-2017-fv3.2.zarr"
dataset = make_cmip7(variable="prw", overrides={"source_name": source_name}, seed=7)
dataset = dataset.isel(lat=slice(None, None, -1))
dataset = dataset.assign_coords(bnds=[0, 1])
dataset["lat_bnds"] = (
    ("lat", "bnds"),
    np.column_stack([dataset["lat"].values - 0.5, dataset["lat"].values + 0.5]),
)

dataset

Load the ESA CCI fix-plan document. The `source_name` metadata lets the plan store select the zarr-style plan.

In [ ]:
plan_path = Path("notebooks/plans/esa_cci_water_vapour_plan.json")
if not plan_path.exists():
    plan_path = Path("plans/esa_cci_water_vapour_plan.json")

plan_path

In [ ]:
findings = woodpecker.check_plan(plan_path, inputs=dataset)
findings.fix_ids

A dry run reports what would change but leaves the dataset untouched.

In [ ]:
dry_run = woodpecker.fix_plan(plan_path, inputs=dataset, write=False)

dry_run.stats, tuple(dataset.data_vars), tuple(dataset.dims)

Apply the same plan in memory with `write=True`.

In [ ]:
write = woodpecker.fix_plan(plan_path, inputs=dataset, write=True)

(
    write.stats,
    tuple(dataset.data_vars),
    tuple(dataset.dims),
    dataset.attrs["realm"],
    dataset.attrs["branded_variable"],
    float(dataset["lat"].values[0]) < float(dataset["lat"].values[-1]),
)

In [ ]:
recheck = woodpecker.check_plan(plan_path, inputs=dataset)
recheck.has_findings